# 06. Weak Supervision NER

Построение BIO-разметки поисковых запросов M.Video из словарей брендов/категорий и регулярных атрибутов (цвет, память, размер).

**Цель:** получить слаборазмеченный датасет для обучения CRF NER без ручной разметки.


In [ ]:
import sys
from pathlib import Path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (8, 4)


In [ ]:
from pathlib import Path
from collections import Counter
import pyarrow.parquet as pq
import pandas as pd

from src.ner.labeling import WeakLabeler, bio_to_entities

DATA = ROOT / 'файлы' / 'query_clicks.parquet'
ART = ROOT / 'artifacts'
ART.mkdir(exist_ok=True)
FIG = ROOT / 'figures'
FIG.mkdir(exist_ok=True)


## Словари брендов и категорий


In [ ]:
brands_p, cats_p = ART / 'brands.txt', ART / 'categories.txt'
if not brands_p.exists() or not cats_p.exists():
    import subprocess
    subprocess.check_call([
        sys.executable, str(ROOT / 'scripts' / 'build_dictionaries.py'),
        '--max-rows', '300000',
    ])
print('brands:', len(brands_p.read_text(encoding='utf-8').splitlines()))
print('categories:', len(cats_p.read_text(encoding='utf-8').splitlines()))
print('top brands:', brands_p.read_text(encoding='utf-8').splitlines()[:10])


## Выборка запросов


In [ ]:
pf = pq.ParquetFile(DATA)
parts = []
for i in range(min(3, pf.num_row_groups)):
    t = pf.read_row_group(i, columns=['toValidUTF8(query_text)', 'toValidUTF8(sku_brand_name)'])
    df = t.to_pandas()
    df.columns = ['query', 'brand']
    parts.append(df)
df = pd.concat(parts).dropna(subset=['query'])
df['query'] = df['query'].astype(str).str.strip()
df = df[df['query'].str.len() >= 2].drop_duplicates('query').head(20000)
print(len(df), 'уникальных запросов')
df.head()


## BIO-разметка


In [ ]:
labeler = WeakLabeler.from_files(ART / 'brands.txt', ART / 'categories.txt')
examples = []
for q in df['query'].head(5000):
    sent = labeler.label_query(q)
    ents = bio_to_entities(sent, query=q)
    if ents:
        examples.append((q, sent, ents))

print(f'Запросов с ≥1 сущностью (из 5000): {len(examples)}')
for q, sent, ents in examples[:8]:
    print('Q:', q)
    print('BIO:', ' '.join(f'{t}/{tag}' for t, tag in sent))
    print('ENT:', ents)
    print('-' * 60)


## Распределение сущностей


In [ ]:
labels = Counter()
for q in df['query']:
    for _, tag in labeler.label_query(q):
        if tag.startswith('B-'):
            labels[tag[2:]] += 1
print(labels)
fig, ax = plt.subplots()
ax.bar(list(labels.keys()), list(labels.values()), color='#E31E24')
ax.set_title('Распределение сущностей (weak labels)')
fig.savefig(FIG / '20_entity_distribution.png', dpi=140, bbox_inches='tight')
plt.show()
